<a href="https://colab.research.google.com/github/VeerJava2021/nba-game-predictor/blob/main/nba_game_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("wyattowalsh/basketball")
print(path)



100%|██████████| 697M/697M [00:06<00:00, 105MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/wyattowalsh/basketball/versions/238


In [2]:
import pandas as pd
games = pd.read_csv(path + "/csv/game.csv")
games.head(20)


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01 00:00:00,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02 00:00:00,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02 00:00:00,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02 00:00:00,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02 00:00:00,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season
5,21946,1610610026,CLR,Cleveland Rebels,24600006,1946-11-03 00:00:00,CLR vs. HUS,W,0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,60.0,-11,0,Regular Season
6,21946,1610610031,PIT,Pittsburgh Ironmen,24600007,1946-11-04 00:00:00,PIT vs. WAS,L,0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,71.0,15,0,Regular Season
7,21946,1610612738,BOS,Boston Celtics,24600008,1946-11-05 00:00:00,BOS vs. CHS,L,0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,57.0,2,0,Regular Season
8,21946,1610610028,DEF,Detroit Falcons,24600009,1946-11-05 00:00:00,DEF vs. BOM,L,0,18.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,4,0,Regular Season
9,21946,1610610032,PRO,Providence Steamrollers,24600011,1946-11-07 00:00:00,PRO vs. CHS,W,0,31.0,...,NaN,NaN,NaN,NaN,NaN,14.0,65.0,-8,0,Regular Season


In [3]:
games['season_id'] = games['season_id'].astype(str)
games['season'] = games['season_id'].str[1:].astype(int)

recent_games = games[games['season'] >= 1985].copy()
print(recent_games.shape)
recent_games.head()

(46512, 56)


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type,season
19186,21985,1610612737,ATL,Atlanta Hawks,28500005,1985-10-25 00:00:00,ATL vs. WAS,L,240,41.0,...,21.0,11.0,7.0,17.0,19.0,100.0,9,0,Regular Season,1985
19187,21985,1610612758,SAC,Sacramento Kings,28500006,1985-10-25 00:00:00,SAC vs. LAC,L,240,39.0,...,19.0,7.0,7.0,18.0,32.0,108.0,4,0,Regular Season,1985
19188,21985,1610612765,DET,Detroit Pistons,28500010,1985-10-25 00:00:00,DET vs. MIL,W,240,39.0,...,27.0,10.0,7.0,20.0,32.0,116.0,-2,0,Regular Season,1985
19189,21985,1610612762,UTH,Utah Jazz,28500011,1985-10-25 00:00:00,UTH vs. HOU,L,240,42.0,...,23.0,10.0,7.0,19.0,28.0,112.0,4,0,Regular Season,1985
19190,21985,1610612744,GOS,Golden State Warriors,28500008,1985-10-25 00:00:00,GOS vs. DEN,L,240,36.0,...,26.0,11.0,3.0,22.0,40.0,119.0,14,0,Regular Season,1985


In [4]:
# Catergorize wins as 1 and loss as 0
recent_games['home_win'] = (recent_games['wl_home'] == 'W').astype(int)
# Home team rows
home_df = recent_games[['game_id', 'game_date', 'team_abbreviation_home', 'pts_home', 'reb_home', 'ast_home', 'home_win']].copy()
home_df.columns = ['game_id', 'game_date', 'team', 'pts', 'reb', 'ast', 'win']

# Away team rows
away_df = recent_games[['game_id', 'game_date', 'team_abbreviation_away', 'pts_away', 'reb_away', 'ast_away', 'home_win']].copy()
away_df.columns = ['game_id', 'game_date', 'team', 'pts', 'reb', 'ast', 'win']
away_df['win'] = 1 - away_df['win']  # if home team didn't win, away team did

# Stack them into one long table
team_games = pd.concat([home_df, away_df], ignore_index=True)
team_games = team_games.sort_values(['team', 'game_date']).reset_index(drop=True)

print(team_games.shape)
team_games.head(10)

(93024, 7)


,game_id,game_date,team,pts,reb,ast,win
0,12200008,2022-10-02 00:00:00,ADL,134.0,40.0,28.0,1
1,12200025,2022-10-06 00:00:00,ADL,98.0,41.0,22.0,0
2,11200002,2012-10-06 00:00:00,ALB,84.0,36.0,17.0,0
3,11400022,2014-10-08 00:00:00,ALB,94.0,36.0,17.0,1
4,28500005,1985-10-25 00:00:00,ATL,91.0,44.0,25.0,0
5,28500013,1985-10-26 00:00:00,ATL,91.0,36.0,14.0,0
6,28500030,1985-10-29 00:00:00,ATL,102.0,60.0,21.0,1
7,28500045,1985-11-01 00:00:00,ATL,105.0,39.0,22.0,0
8,28500054,1985-11-02 00:00:00,ATL,114.0,50.0,25.0,1
9,28500060,1985-11-05 00:00:00,ATL,113.0,50.0,18.0,0


In [5]:
nba_teams = ['ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW',
             'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK',
             'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']
team_games = team_games[team_games['team'].isin(nba_teams)].reset_index(drop=True)

print(team_games['team'].unique())
team_games.iloc[150:200]

['ATL' 'BKN' 'BOS' 'CHA' 'CHI' 'CLE' 'DAL' 'DEN' 'DET' 'GSW' 'HOU' 'IND'
 'LAC' 'LAL' 'MEM' 'MIA' 'MIL' 'MIN' 'NOP' 'NYK' 'OKC' 'ORL' 'PHI' 'PHX'
 'POR' 'SAC' 'SAS' 'TOR' 'UTA' 'WAS']


,game_id,game_date,team,pts,reb,ast,win
150,28600688,1987-03-07 00:00:00,ATL,122.0,56.0,26.0,1
151,28600699,1987-03-09 00:00:00,ATL,108.0,42.0,25.0,1
152,28600701,1987-03-10 00:00:00,ATL,113.0,54.0,24.0,1
153,28600720,1987-03-13 00:00:00,ATL,113.0,41.0,27.0,1
154,28600737,1987-03-15 00:00:00,ATL,104.0,45.0,23.0,1
155,28600747,1987-03-17 00:00:00,ATL,118.0,56.0,34.0,1
156,28600752,1987-03-18 00:00:00,ATL,107.0,46.0,26.0,0
157,28600761,1987-03-20 00:00:00,ATL,114.0,52.0,27.0,1
158,28600772,1987-03-21 00:00:00,ATL,97.0,43.0,19.0,1
159,28600789,1987-03-24 00:00:00,ATL,96.0,44.0,13.0,1


In [6]:
team_games['pts_last10'] = team_games.groupby('team')['pts'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['reb_last10'] = team_games.groupby('team')['reb'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['ast_last10'] = team_games.groupby('team')['ast'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
team_games['win_pct_last10'] = team_games.groupby('team')['win'].transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())

In [7]:
# Merge home team's rolling stats
recent_games = recent_games.merge(
    team_games[['game_id', 'team', 'pts_last10', 'reb_last10', 'ast_last10', 'win_pct_last10']],
    left_on=['game_id', 'team_abbreviation_home'],
    right_on=['game_id', 'team'],
    how='left'
).rename(columns={'pts_last10': 'home_pts_last10', 'reb_last10': 'home_reb_last10',
                   'ast_last10': 'home_ast_last10', 'win_pct_last10': 'home_winpct_last10'})

# Merge away team's rolling stats
recent_games = recent_games.merge(
    team_games[['game_id', 'team', 'pts_last10', 'reb_last10', 'ast_last10', 'win_pct_last10']],
    left_on=['game_id', 'team_abbreviation_away'],
    right_on=['game_id', 'team'],
    how='left'
).rename(columns={'pts_last10': 'away_pts_last10', 'reb_last10': 'away_reb_last10',
                   'ast_last10': 'away_ast_last10', 'win_pct_last10': 'away_winpct_last10'})

recent_games = recent_games.dropna(subset=['home_pts_last10', 'away_pts_last10'])

In [8]:
features = ['home_pts_last10', 'away_pts_last10', 'home_reb_last10', 'away_reb_last10',
            'home_ast_last10', 'away_ast_last10', 'home_winpct_last10', 'away_winpct_last10']

X = recent_games[features]
y = recent_games['home_win']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score
preds = model.predict(X_test)
print(f"Model accuracy: {accuracy_score(y_test, preds):.2%}")
print(f"Baseline (always home): {y_test.mean():.2%}")

Model accuracy: 64.80%
Baseline (always home): 60.24%


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
def get_team_stats_before(team, date, team_games_df):
    team_games['game_date'] = pd.to_datetime(team_games['game_date'])
    recent_games['game_date'] = pd.to_datetime(recent_games['game_date'])
    date = pd.Timestamp(date)
    row = team_games_df[(team_games_df['team'] == team) & (team_games_df['game_date'] <= date)].tail(1)
    if row.empty:
        return None
    return row[['pts_last10', 'reb_last10', 'ast_last10', 'win_pct_last10']].values[0]

def predict_matchup(team_a, date_a, team_b, date_b, model, team_games_df):
    stats_a = get_team_stats_before(team_a, date_a, team_games_df)
    stats_b = get_team_stats_before(team_b, date_b, team_games_df)

    if stats_a is None or stats_b is None:
        return "Not enough data for one of these teams/dates"

    features_row = [[stats_a[0], stats_b[0], stats_a[1], stats_b[1], stats_a[2], stats_b[2], stats_a[3], stats_b[3]]]
    prob = model.predict_proba(features_row)[0]

    print(f"{team_a} win probability: {prob[1]:.1%}")
    print(f"{team_b} win probability: {prob[0]:.1%}")

# Example usage:
predict_matchup('GSW', '2016-04-01', 'CLE', '2016-04-01', model, team_games)

GSW win probability: 78.3%
CLE win probability: 21.7%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
